In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [48]:
df = pd.read_csv('../Loan_Data.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Shape: (614, 13)

First 5 rows:


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [49]:
print("=== Data Types & Non-Null Counts ===")
print(df.info())

print("\n=== Missing Value Counts ===")
print(df.isnull().sum())

print("\n=== Statistical Summary ===")
df.describe()

=== Data Types & Non-Null Counts ===
<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB
None

=== Missing Value Counts ===
Loan_ID               0
Gender               13
Married

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [50]:
# Cell 5 — Data Cleaning & Imputation

# Drop Loan_ID
df = df.drop(columns=['Loan_ID'], errors='ignore')
print("✅ Dropped Loan_ID")

# Fill categorical columns with MODE
cat_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed']
for col in cat_cols:
    most_frequent = df[col].mode()[0]
    df[col] = df[col].fillna(most_frequent)
    print(f"✅ {col} filled with mode: '{most_frequent}'")

# Fill numeric columns with MEDIAN
num_cols = ['LoanAmount', 'Loan_Amount_Term', 'Credit_History']
for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"✅ {col} filled with median: {median_val}")

# Confirm all zeros
print("\n=== Missing Values After Cleaning ===")
print(df.isnull().sum())
print("\nShape:", df.shape)

✅ Dropped Loan_ID
✅ Gender filled with mode: 'Male'
✅ Married filled with mode: 'Yes'
✅ Dependents filled with mode: '0'
✅ Self_Employed filled with mode: 'No'
✅ LoanAmount filled with median: 128.0
✅ Loan_Amount_Term filled with median: 360.0
✅ Credit_History filled with median: 1.0

=== Missing Values After Cleaning ===
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

Shape: (614, 12)


In [51]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nColumns:", df.columns.tolist())

Shape: (614, 12)

Missing values:
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

Columns: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


In [52]:
# Cell 6 — Feature Engineering

# Total income of applicant + co-applicant
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# Log transform to reduce right skew (income data has outliers)
df['Log_TotalIncome'] = np.log1p(df['TotalIncome'])
df['Log_LoanAmount'] = np.log1p(df['LoanAmount'])

# EMI = monthly repayment estimate
df['EMI'] = df['LoanAmount'] / df['Loan_Amount_Term']

# Balance income after EMI deduction
df['BalanceIncome'] = df['TotalIncome'] - (df['EMI'] * 1000)

print("✅ New features created")
print("\nNew columns added:")
print(df[['TotalIncome', 'Log_TotalIncome', 'Log_LoanAmount', 'EMI', 'BalanceIncome']].head())
print("\nNew shape:", df.shape)

✅ New features created

New columns added:
   TotalIncome  Log_TotalIncome  Log_LoanAmount       EMI  BalanceIncome
0       5849.0         8.674197        4.859812  0.355556    5493.444444
1       6091.0         8.714732        4.859812  0.355556    5735.444444
2       3000.0         8.006701        4.204693  0.183333    2816.666667
3       4941.0         8.505525        4.795791  0.333333    4607.666667
4       6000.0         8.699681        4.955827  0.391667    5608.333333

New shape: (614, 17)


In [53]:
# Cell 7 — Encoding Categorical Variables

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# Binary encode (only 2 unique values each)
binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed']
for col in binary_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f"✅ {col} encoded: {df[col].unique()}")

# Fix Dependents — '3+' needs to become numeric 3
df['Dependents'] = df['Dependents'].replace('3+', '3').astype(int)
print(f"✅ Dependents encoded: {df['Dependents'].unique()}")

# One-hot encode Property_Area (3 categories: Urban, Rural, Semiurban)
df = pd.get_dummies(df, columns=['Property_Area'], drop_first=True)
print(f"✅ Property_Area one-hot encoded")

# Encode target column Loan_Status (Y=1, N=0)
df['Loan_Status'] = le.fit_transform(df['Loan_Status'])
print(f"✅ Loan_Status encoded: {df['Loan_Status'].unique()}")

print("\nShape after encoding:", df.shape)
print("\nAll columns:", df.columns.tolist())

✅ Gender encoded: [1 0]
✅ Married encoded: [0 1]
✅ Education encoded: [0 1]
✅ Self_Employed encoded: [0 1]
✅ Dependents encoded: [0 1 2 3]
✅ Property_Area one-hot encoded
✅ Loan_Status encoded: [1 0]

Shape after encoding: (614, 18)

All columns: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Loan_Status', 'TotalIncome', 'Log_TotalIncome', 'Log_LoanAmount', 'EMI', 'BalanceIncome', 'Property_Area_Semiurban', 'Property_Area_Urban']


In [54]:
# Cell 8 — Feature Selection & Train/Test Split

from sklearn.model_selection import train_test_split

# Drop raw columns replaced by engineered features
drop_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'TotalIncome']
df = df.drop(columns=drop_cols, errors='ignore')
print("✅ Dropped raw columns, new shape:", df.shape)

# Separate features (X) and target (y)
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

print("\n✅ Features (X) shape:", X.shape)
print("✅ Target  (y) shape:", y.shape)
print("\nFeature columns:", X.columns.tolist())
print("\nClass distribution:")
print(y.value_counts())

# Stratified split — preserves Y/N ratio in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\n✅ Train size:", X_train.shape)
print("✅ Test  size:", X_test.shape)

✅ Dropped raw columns, new shape: (614, 14)

✅ Features (X) shape: (614, 13)
✅ Target  (y) shape: (614,)

Feature columns: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Loan_Amount_Term', 'Credit_History', 'Log_TotalIncome', 'Log_LoanAmount', 'EMI', 'BalanceIncome', 'Property_Area_Semiurban', 'Property_Area_Urban']

Class distribution:
Loan_Status
1    422
0    192
Name: count, dtype: int64

✅ Train size: (491, 13)
✅ Test  size: (123, 13)


In [55]:
# Cell 9 — Feature Scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training data — then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Scaling done")
print("\nTrain mean (should be ~0):", X_train_scaled.mean(axis=0).round(2))
print("\nTrain std  (should be ~1):", X_train_scaled.std(axis=0).round(2))

✅ Scaling done

Train mean (should be ~0): [ 0.  0.  0. -0. -0. -0. -0.  0. -0. -0. -0. -0.  0.]

Train std  (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [56]:
# Cell 10 — Train Logistic Regression Model

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced',
    random_state=42
)

model.fit(X_train_scaled, y_train)

print("✅ Model trained successfully!")
print("\nModel parameters:")
print(f"  Solver       : {model.solver}")
print(f"  Max iter     : {model.max_iter}")
print(f"  Class weight : {model.class_weight}")
print(f"  Iterations   : {model.n_iter_[0]} (converged)")

✅ Model trained successfully!

Model parameters:
  Solver       : lbfgs
  Max iter     : 1000
  Class weight : balanced
  Iterations   : 12 (converged)
